# YOLO 특강 1회차 실습 B — 뇌종양 MRI/CT 탐지: 클래스가 둘이 되면 보이는 것

**파일**: `04_yolo_day1_brain_tumor_lab_v1.0.ipynb` · **환경**: Google Colab (T4 GPU) · **모델**: YOLO26n · **데이터**: Ultralytics Brain-tumor (학습 893장 · 검증 223장 · 클래스 `negative`, `positive`)

알약 실습은 클래스가 하나라 **클래스별 AP**와 **혼동 행렬**을 볼 수 없었습니다. 이 노트북은 그 두 가지를 보는 것이 목적입니다.

| 장 | 내용 | 슬라이드 |
|---|---|---|
| 1 | 데이터 내려받기와 클래스 분포 | 33 |
| 2 | 학습 (15 에포크, 약 6분) | 33 |
| 3 | 평가: 클래스별 AP와 혼동 행렬 읽기 | 23, 34 |
| 4 | 임계값과 비용: 놓친 종양 vs 과잉 경보 | 29, 34 |
| 5 | 분할 점검: 환자 단위로 나눌 수 있는가 | 32, 34 |

> 알약 노트북 1장(설치)을 이미 실행한 런타임에서 이어서 실행하면 설치 셀은 건너뛰어도 됩니다.

## 1장. 데이터 내려받기와 클래스 분포

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics.data.utils import check_det_dataset
from pathlib import Path
from collections import Counter
import numpy as np

data = check_det_dataset("brain-tumor.yaml")      # 처음 실행 시 약 4MB 다운로드
ROOT = Path(data["train"]).parent.parent
NAMES = data["names"]
print("클래스:", NAMES)

for split in ["train", "val"]:
    imgs = sorted((ROOT / "images" / split).glob("*"))
    lbls = sorted((ROOT / "labels" / split).glob("*.txt"))
    cnt = Counter(); per_img = []
    for p in lbls:
        rows = [l for l in p.read_text().splitlines() if l.strip()]
        per_img.append(len(rows))
        cnt.update(NAMES[int(l.split()[0])] for l in rows)
    print(f"{split:5s} 이미지 {len(imgs)}장 · 라벨 파일 {len(lbls)}개 (라벨 없는 이미지 {len(imgs)-len(lbls)}장) · 이미지당 박스 {np.mean(per_img):.1f}개 · 클래스별 박스 {dict(cnt)}")

**읽어 볼 점**
- 학습과 검증의 클래스 비율이 다릅니다. 검증 세트는 `negative`가 더 많습니다. 어느 쪽이 현장 분포에 가까울까요?
- `negative`(종양 없음)도 박스로 라벨되어 있습니다. 보통은 물체가 없는 사진을 빈 라벨(배경)로 두는데, 이 데이터는 "종양이 아닌 것"을 박스로 쳤습니다. 우리 프로젝트라면 어떻게 정할지가 라벨 기준서의 첫 줄입니다.
- 라벨 파일이 없는 이미지가 몇 장 있습니다. 이것이 배경(물체 없음) 이미지로 학습에 쓰입니다.

In [ ]:
# 사진 몇 장 보기: 정답 박스와 클래스
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show_gt(split, n=4):
    lbls = sorted((ROOT / "labels" / split).glob("*.txt"))[:n]
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    for ax, p in zip(axes, lbls):
        img = next((ROOT / "images" / split).glob(p.stem + ".*"))
        im = Image.open(img); W, H = im.size
        ax.imshow(im, cmap="gray")
        for line in p.read_text().splitlines():
            if not line.strip(): continue
            c, cx, cy, w, h = map(float, line.split())
            color = "lime" if int(c) == 1 else "deepskyblue"
            ax.add_patch(patches.Rectangle(((cx-w/2)*W, (cy-h/2)*H), w*W, h*H, fill=False, color=color, lw=2))
            ax.text((cx-w/2)*W, (cy-h/2)*H-3, NAMES[int(c)], color=color, fontsize=9)
        ax.set_title(p.stem); ax.axis("off")
    plt.show()

show_gt("train")

## 2장. 학습

알약과 같은 출발점(`yolo26n.pt`)입니다. 의료 영상은 좌우 뒤집기가 의미를 바꾸지 않으므로 기본값(`fliplr=0.5`)을 그대로 둡니다. 회전·색 변화는 넣지 않습니다(MRI/CT는 촬영 조건이 고정돼 있습니다).

In [ ]:
from ultralytics import YOLO

EPOCHS = 15
IMGSZ = 640

model = YOLO("yolo26n.pt")
results = model.train(
    data="brain-tumor.yaml",
    epochs=EPOCHS, patience=10,
    imgsz=IMGSZ, batch=16,
    seed=0,
    project="runs_bt", name="yolo26n", exist_ok=True,
)
RUN = Path(results.save_dir)
print("결과 폴더:", RUN)

## 3장. 평가 — 클래스별 AP와 혼동 행렬

`mAP`는 클래스별 AP의 평균입니다. 평균이 괜찮아도 한 클래스만 낮다면 그 클래스의 데이터가 부족하거나 다른 클래스와 헷갈리고 있다는 신호입니다.

In [ ]:
best = YOLO(RUN / "weights" / "best.pt")
m = best.val(data="brain-tumor.yaml", imgsz=IMGSZ, nms=True, plots=True, verbose=False)
print(f"전체 mAP50 {m.box.map50:.3f} · mAP50-95 {m.box.map:.3f}")
for i, cls_idx in enumerate(m.box.ap_class_index):
    print(f"  {NAMES[int(cls_idx)]:9s} AP50 {m.box.ap50[i]:.3f} · AP50-95 {m.box.ap[i]:.3f} · precision {m.box.p[i]:.3f} · recall {m.box.r[i]:.3f}")

### 혼동 행렬은 신뢰도 기준을 정하고 읽습니다
위의 `val()`은 mAP를 위해 신뢰도 0.001부터 모든 예측을 모읍니다. 그 상태의 혼동 행렬은 `background` 칸(헛탐지)이 수만 개로 부풀어 읽을 수 없습니다. 혼동 행렬은 **실제 서비스에서 쓸 기준(예: 0.25)**으로 다시 계산해서 봅니다.

In [ ]:
# 혼동 행렬: 세로(행) = 모델 예측, 가로(열) = 실제 정답. 마지막 'background'는 물체 없음.
#  - background 열의 값: 정답이 없는 곳에 박스를 그림 (헛탐지, FP)
#  - background 행의 값: 정답이 있는데 못 찾음 (놓침, FN)
import pandas as pd
m25 = best.val(data="brain-tumor.yaml", imgsz=IMGSZ, nms=True, conf=0.25, plots=True, verbose=False)
labels = [NAMES[i] for i in range(len(NAMES))] + ["background"]
cm = pd.DataFrame(m25.confusion_matrix.matrix.astype(int), index=[f"pred:{l}" for l in labels], columns=[f"true:{l}" for l in labels])
cm

In [ ]:
Image.open(Path(m25.save_dir) / "confusion_matrix.png").resize((700, 600))

**읽는 순서**
1. 대각선(맞음)보다 큰 비대각 칸이 있는가?
2. `true:positive` 열에서 `pred:negative`와 `pred:background` 값 — **종양을 놓친 수**입니다. 의료에서는 이 칸이 가장 비쌉니다.
3. `true:negative` 열의 `pred:positive` — 과잉 경보입니다. 사람이 다시 보는 흐름이라면 어느 정도 감수할 수 있습니다.

## 4장. 임계값과 비용 — 놓친 종양 vs 과잉 경보

`conf`를 바꾸면 positive를 놓치는 수(FN)와 헛탐지 수(FP)가 반대로 움직입니다. 알약 실습의 "어느 쪽 실수가 더 비쌀까"를 의료 데이터에서 숫자로 봅니다.

In [ ]:
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1]); ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

def gt_boxes(stem, W, H):
    p = ROOT / "labels" / "val" / f"{stem}.txt"
    out = []
    if p.exists():
        for line in p.read_text().splitlines():
            if line.strip():
                c, cx, cy, w, h = map(float, line.split())
                out.append((int(c), ((cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H)))
    return out

val_imgs = sorted((ROOT / "images" / "val").glob("*"))
POS = 1
rows = []
for conf in (0.1, 0.25, 0.5, 0.75):
    rs = best.predict(val_imgs, conf=conf, nms=True, imgsz=IMGSZ, verbose=False)
    tp = fp = fn = 0
    for p, r in zip(val_imgs, rs):
        H, W = r.orig_shape
        gts = [b for c, b in gt_boxes(p.stem, W, H) if c == POS]
        preds = [tuple(b) for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls) if int(c) == POS]
        used = set()
        for pb in preds:
            j = next((k for k, g in enumerate(gts) if k not in used and iou(pb, g) >= 0.5), None)
            if j is None: fp += 1
            else: used.add(j); tp += 1
        fn += len(gts) - len(used)
    rows.append((conf, tp, fp, fn, tp/(tp+fn) if tp+fn else 0, tp/(tp+fp) if tp+fp else 0))
pd.DataFrame(rows, columns=["conf", "TP", "FP(과잉 경보)", "FN(놓친 종양)", "recall", "precision"]).round(3)

**질문**: 사람이 최종 확인하는 보조 시스템이라면 `conf`를 어디에 두겠습니까? 반대로 사람 없이 자동으로 걸러내는 시스템이라면? 같은 모델인데 답이 달라집니다 — 신뢰도 기준은 모델이 아니라 서비스가 정합니다.

## 5장. 분할 점검 — 이 검증 점수를 믿어도 될까

한 환자의 여러 단면이 학습과 검증에 나뉘어 들어가면 검증 점수가 부풀려집니다. 이 데이터셋은 학습 파일명에 ID처럼 보이는 앞자리가 있지만 **검증 파일명에는 없어** 겹침을 확인할 수 없습니다. 공개 데이터를 쓸 때 가장 먼저 물어야 할 질문입니다.

In [ ]:
# 파일 이름에 환자·제품을 뜻하는 접두사가 있는지 봅니다.
train_names = [p.name for p in sorted((ROOT / "images" / "train").glob("*"))]
val_names = [p.name for p in sorted((ROOT / "images" / "val").glob("*"))]
print("train 파일명 예:", train_names[:5])
print("val   파일명 예:", val_names[:5])

prefix = Counter(n.split("_")[0] for n in train_names)
print(f"train 접두사 종류 {len(prefix)}개 · 한 접두사당 최대 {max(prefix.values())}장 · 예: {prefix.most_common(3)}")
# 학습 파일명의 앞자리(00054 …)는 환자·케이스 ID처럼 보이고 같은 ID의 단면이 여러 장 있습니다.
# 검증 파일명은 'val_1 (n)' 형식이라 ID가 없습니다.
# 질문: 검증 세트에 학습 세트와 같은 환자가 들어 있는지 알 수 있나요? 이 점수를 '새 환자'에 대한 성능이라고 말할 수 있을까요?

In [ ]:
# 우리 데이터에 환자(제품·로트) ID가 있다면: 그룹 단위로 나누는 템플릿
# 예: 파일명이 P013_slice_07.png 처럼 앞자리가 환자 ID인 경우
from sklearn.model_selection import GroupShuffleSplit

files = [f"P{pid:03d}_slice_{s:02d}.png" for pid in range(1, 21) for s in range(1, 6)]   # 가상 파일 100개 (환자 20명 × 5장)
groups = [f.split("_")[0] for f in files]                                                   # 환자 ID

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, val_idx = next(gss.split(files, groups=groups))
train_pat = {groups[i] for i in train_idx}; val_pat = {groups[i] for i in val_idx}
print(f"train 환자 {len(train_pat)}명 · val 환자 {len(val_pat)}명 · 겹치는 환자 {len(train_pat & val_pat)}명")
# 실제 데이터에서는 files를 자기 이미지 목록으로, groups를 자기 ID 규칙으로 바꾸고, 결과대로 images/labels 폴더를 나눕니다.

In [ ]:
# 2회차를 위해 학습 결과를 Google Drive에 저장 (Colab 전용)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    dst = Path("/content/drive/MyDrive/yolo_bootcamp/day1_brain_tumor"); dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(RUN / "weights" / "best.pt", dst / "best.pt")
    shutil.copy(Path(m25.save_dir) / "confusion_matrix.png", dst / "confusion_matrix.png")
    print("저장 완료:", dst)
except ImportError:
    print(f"Colab이 아닙니다. {RUN / 'weights' / 'best.pt'} 를 직접 보관하세요.")

## 정리

- 클래스가 둘 이상이면 **클래스별 AP**부터 본다 — 평균에 가려진 약한 클래스를 찾는다
- 혼동 행렬은 세로가 예측, 가로가 정답 — `background` 행·열이 놓침과 헛탐지다
- 어느 칸의 실수가 비싼지는 도메인이 정한다 — 의료는 놓침(FN)이 비싸다
- 환자·제품 단위로 나눌 수 없는 데이터의 검증 점수는 조심해서 읽는다